# MOMENT SAE — Sparse Autoencoder Interpretability
Train a ReLU SAE on layer 18 of MOMENT-1-large to find monosemantic features for frequency, trend, and noise.

In [ ]:
# momentfm pins huggingface-hub==0.24.0 which conflicts with Colab numpy 2.x
# Install with --no-deps since Colab already has compatible versions
!pip install momentfm --no-deps --quiet
!pip install transformers einops --quiet

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/mzmyslowski-professional/moment_sae.git'
REPO_DIR = '/content/moment_sae'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Repo updated.')
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Repo cloned.')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/moment_sae'
REPO_DIR   = '/content/moment_sae'

# Code lives in the repo; only large artifacts persist on Drive
for name in ('data', 'checkpoints', 'outputs'):
    drive_path = os.path.join(DRIVE_BASE, name)
    repo_path  = os.path.join(REPO_DIR, name)
    os.makedirs(drive_path, exist_ok=True)
    if not os.path.exists(repo_path):
        os.symlink(drive_path, repo_path)
        print(f'Linked {repo_path} -> {drive_path}')
    else:
        print(f'{repo_path} already present')

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
!pytest tests/ -v 2>&1 | tail -20

## Step 1: Extract Activations
Generates synthetic time series and runs them through frozen MOMENT-1-large.
Saves ~1.25 GB to `data/activations.npy` on Drive. Skipped if already exists.

In [ ]:
import os
if os.path.exists('data/activations.npy'):
    print('Activations already exist — skipping extraction.')
else:
    %run generate_activations.py

## Step 2: Train SAE
Trains for 50,000 steps (~25–30 min on T4). Checkpoints saved every 10,000 steps to Drive.
Re-running this cell resumes from the latest checkpoint.

In [ ]:
%run train_sae.py

## Step 3: Visualize Features
Generates per-feature top-20 plots, summary dashboard, and selectivity table.
Results saved to `outputs/` on Drive.

In [ ]:
%run visualize.py

In [ ]:
import pandas as pd
table = pd.read_csv('outputs/selectivity_table.csv')
print(f'Total live features: {len(table)}')
print(f'Monosemantic candidates: {(table["dominant_axis"] != "none").sum()}')
print('\nTop monosemantic features by dominant axis:')
print(table[table['dominant_axis'] != 'none']
      .sort_values('freq_sel', ascending=False)
      .head(20)
      .to_string(index=False))

In [ ]:
from IPython.display import Image, display
import glob

plots = sorted(glob.glob('outputs/feature_*_top20.png'))
if plots:
    display(Image(plots[0]))
    print(f'Showing: {plots[0]}')